# Question 1: Random Forest Setup

Build the modeling dataset for predicting dog breed popularity tier from breed traits.

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

for base_dir in [Path.cwd(), *Path.cwd().parents]:
    data_dir = base_dir / 'data'
    if data_dir.exists():
        break
else:
    raise FileNotFoundError('Could not find a data directory.')

def find_data_file(file_name):
    for subdir in ['interim', 'processed']:
        path = data_dir / subdir / file_name
        if path.exists():
            return path
    raise FileNotFoundError(f'Could not find {file_name} in data/interim or data/processed.')

## Load Cleaned CSVs

In [ ]:
breed_traits = pd.read_csv(find_data_file('breed_traits.csv'))
breed_rank = pd.read_csv(find_data_file('breed_ranks.csv'))

## Create Popularity Target

In [ ]:
rank_cols = [f'{year} Rank' for year in range(2013, 2026)]
rank_values = breed_rank[rank_cols].to_numpy(dtype=float)

average_rank = np.nanmean(rank_values, axis=1)

breed_rank['Popularity Tier'] = pd.cut(
    average_rank,
    bins=[0,25, 50, 100, np.inf],
    labels=['Very Popular', 'Popular', 'Somewhat Popular', 'Not Popular'],
    include_lowest=True
)

breed_rank.head()

## Clean And Engineer Predictor Features

In [ ]:
breed_traits_q1 = breed_traits.copy()

categorical_trait_cols = ['Coat Type', 'Coat Length']
numeric_trait_cols = [
    'Affectionate With Family',
    'Good With Young Children',
    'Good With Other Dogs',
    'Shedding Level',
    'Coat Grooming Frequency',
    'Drooling Level',
    'Openness To Strangers',
    'Playfulness Level',
    'Watchdog/Protective Nature',
    'Adaptability Level',
    'Trainability Level',
    'Energy Level',
    'Barking Level',
    'Mental Stimulation Needs'
]

breed_traits_q1[numeric_trait_cols] = breed_traits_q1[numeric_trait_cols].apply(pd.to_numeric, errors='coerce')

model_features = pd.concat(
    [
        breed_traits_q1[['Breed']],
        breed_traits_q1[numeric_trait_cols],
        breed_traits_q1[categorical_trait_cols],
        breed_rank[['Popularity Tier']]
    ],
    axis=1
)

class_col = 'Popularity Tier'

domain_codes = []
for col in model_features.columns:
    if col == 'Breed':
        domain_codes.append(-1)
    elif col in numeric_trait_cols:
        domain_codes.append(0)
    else:
        domain_codes.append(model_features[col].nunique())

domain_code_row = pd.DataFrame(
    [domain_codes],
    columns=model_features.columns
)

class_variable_row = pd.DataFrame(
    [[class_col] + [''] * (len(model_features.columns) - 1)],
    columns=model_features.columns
)

model_features_with_domain_codes = pd.concat(
    [domain_code_row, class_variable_row, model_features],
    ignore_index=True
)

model_features_with_domain_codes